In [ ]:
!pip install pandas matplotlib seaborn autogen-agentchat google-generativeai

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Any, Optional
import autogen_agentchat as autogen
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import google.generativeai as genai
import warnings
warnings.filterwarnings('ignore')

# Configuration for Gemini
class GeminiConfig:
    def __init__(self, model_name: str = "gemini-1.5-flash"):
        self.model_name = model_name

    def get_config(self) -> Dict[str, Any]:
        return {
            "model": self.model_name,
            "api_key": os.getenv("GEMINI_API_KEY", ""),
            "api_type": "gemini"
        }

class EDAMultiAgentSystem:
    def __init__(self):
        self.gemini_config = GeminiConfig()
        self.agents = {}
        self.data = None
        self.results = {}
        self.setup_agents()

    def setup_agents(self):
        """Initialize agents with specific roles."""
        base_config = {
            "config_list": [self.gemini_config.get_config()],
            "temperature": 0.2,
            "timeout": 200,
        }

        # Data Preparation Agent
        self.agents['data_prep'] = AssistantAgent(
            name="DataPrepAgent",
            system_message="""Specialized in data cleaning and preprocessing:
            - Load and inspect datasets
            - Handle missing values and outliers
            - Optimize data types
            - Perform basic feature engineering
            Provide clear explanations for all preprocessing steps.""",
            llm_config=base_config
        )

        # EDA Agent
        self.agents['eda'] = AssistantAgent(
            name="EDAAgent",
            system_message="""Specialized in exploratory data analysis:
            - Generate descriptive statistics
            - Create visualizations
            - Identify trends and correlations
            - Provide actionable insights
            Ensure visualizations are clear and insightful.""",
            llm_config=base_config
        )

        # Report Generator Agent
        self.agents['report'] = AssistantAgent(
            name="ReportAgent",
            system_message="""Specialized in creating EDA reports:
            - Summarize key findings
            - Create executive summaries
            - Organize visualizations
            - Provide recommendations
            Generate professional, accessible reports.""",
            llm_config=base_config
        )

        # Admin Agent
        self.agents['admin'] = UserProxyAgent(
            name="AdminAgent",
            system_message="""Oversees EDA workflow:
            - Coordinates agents
            - Ensures alignment with goals
            - Manages communication
            Maintain focus on comprehensive EDA.""",
            human_input_mode="NEVER",
            max_consecutive_auto_reply=3,
            code_execution_config={"work_dir": "eda_output", "use_docker": False}
        )

    def load_data(self, data_path: str) -> pd.DataFrame:
        """Load data from various formats."""
        try:
            if data_path.endswith('.csv'):
                self.data = pd.read_csv(data_path)
            elif data_path.endswith(('.xlsx', '.xls')):
                self.data = pd.read_excel(data_path)
            elif data_path.endswith('.json'):
                self.data = pd.read_json(data_path)
            else:
                raise ValueError("Unsupported file format")
            print(f"Data loaded: {self.data.shape}")
            return self.data
        except Exception as e:
            print(f"Error loading data: {e}")
            return None

    def create_sample_data(self) -> pd.DataFrame:
        """Generate sample data for testing."""
        np.random.seed(42)
        n_samples = 500
        self.data = pd.DataFrame({
            'age': np.random.randint(18, 75, n_samples),
            'salary': np.random.normal(60000, 20000, n_samples),
            'experience': np.random.randint(0, 35, n_samples),
            'rating': np.random.uniform(1, 5, n_samples),
            'department': np.random.choice(['IT', 'Finance', 'Marketing'], n_samples),
            'location': np.random.choice(['London', 'Paris', 'Berlin'], n_samples)
        })
        # Add missing values and outliers
        self.data.loc[np.random.choice(n_samples, 25), 'rating'] = np.nan
        self.data.loc[np.random.choice(n_samples, 10), 'salary'] = np.random.uniform(150000, 300000, 10)
        print(f"Sample data created: {self.data.shape}")
        return self.data

    def run_data_preparation(self) -> Dict[str, Any]:
        """Execute data preparation phase."""
        print("Starting Data Preparation...")
        if self.data is None:
            self.create_sample_data()

        prep_results = {
            "info": self.get_data_info(),
            "missing": self.handle_missing_values(),
            "outliers": self.detect_outliers()
        }
        self.results['data_preparation'] = prep_results
        print("Data Preparation completed.")
        return prep_results

    def get_data_info(self) -> Dict[str, Any]:
        """Retrieve dataset information."""
        return {
            "shape": self.data.shape,
            "columns": list(self.data.columns),
            "dtypes": self.data.dtypes.to_dict(),
            "missing_counts": self.data.isnull().sum().to_dict()
        }

    def handle_missing_values(self) -> Dict[str, Any]:
        """Handle missing values."""
        missing_summary = self.data.isnull().sum()
        for column in self.data.columns:
            if self.data[column].isnull().any():
                if self.data[column].dtype in ['int64', 'float64']:
                    self.data[column].fillna(self.data[column].mean(), inplace=True)
                else:
                    self.data[column].fillna(self.data[column].mode()[0], inplace=True)
        return {
            "missing_counts": missing_summary.to_dict(),
            "imputation_strategy": "mean for numeric, mode for categorical"
        }

    def detect_outliers(self) -> Dict[str, Any]:
        """Detect outliers using IQR method."""
        numeric_columns = self.data.select_dtypes(include=[np.number]).columns
        outlier_info = {}
        for column in numeric_columns:
            Q1, Q3 = self.data[column].quantile([0.25, 0.75])
            IQR = Q3 - Q1
            bounds = (Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
            outliers = self.data[column][(self.data[column] < bounds[0]) | (self.data[column] > bounds[1])].count()
            outlier_info[column] = {"count": outliers, "bounds": bounds}
        return outlier_info

    def run_eda_analysis(self) -> Dict[str, Any]:
        """Perform EDA analysis."""
        print("Starting EDA Analysis...")
        eda_results = {
            "stats": self.get_descriptive_statistics(),
            "correlations": self.analyze_correlations(),
            "insights": self.generate_insights()
        }
        self.results['eda_analysis'] = eda_results
        print("EDA Analysis completed.")
        return eda_results

    def get_descriptive_statistics(self) -> Dict[str, Any]:
        """Generate descriptive statistics."""
        return {
            "numeric": self.data.describe().to_dict(),
            "categorical": self.data.describe(include=['object', 'category']).to_dict()
        }

    def analyze_correlations(self) -> Dict[str, Any]:
        """Analyze correlations."""
        numeric_data = self.data.select_dtypes(include=[np.number])
        corr_matrix = numeric_data.corr()
        return {"correlation_matrix": corr_matrix.to_dict()}

    def generate_insights(self) -> List[str]:
        """Generate key insights."""
        insights = []
        if self.data.isnull().sum().sum() > 0:
            insights.append("Dataset contains missing values.")
        for column in self.data.select_dtypes(include=[np.number]).columns:
            if abs(self.data[column].skew()) > 1.5:
                insights.append(f"{column} shows significant skewness.")
        return insights

    def create_visualizations(self) -> Dict[str, str]:
        """Generate visualizations."""
        print("Creating visualizations...")
        plt.style.use('ggplot')
        os.makedirs('eda_output', exist_ok=True)
        visualizations = {}

        # Correlation heatmap
        numeric_data = self.data.select_dtypes(include=[np.number])
        if len(numeric_data.columns) > 1:
            plt.figure(figsize=(8, 6))
            sns.heatmap(numeric_data.corr(), annot=True, cmap='viridis')
            plt.title('Correlation Heatmap')
            plt.savefig('eda_output/corr_heatmap.png', dpi=300)
            plt.close()
            visualizations['heatmap'] = 'corr_heatmap.png'

        # Histograms
        for column in numeric_data.columns[:3]:
            plt.figure(figsize=(6, 4))
            sns.histplot(self.data[column], bins=20, color='teal')
            plt.title(f'Distribution of {column}')
            plt.savefig(f'eda_output/hist_{column}.png', dpi=300)
            plt.close()
            visualizations[f'hist_{column}'] = f'hist_{column}.png'

        return visualizations

    def generate_report(self) -> str:
        """Generate EDA report."""
        print("Generating report...")
        report = f"""
# EDA Report

## Overview
- Shape: {self.data.shape}
- Missing Values: {self.data.isnull().sum().sum()}

## Key Findings
{self._format_insights()}

## Visualizations
{self._format_visualizations()}
"""
        with open('eda_output/eda_report.md', 'w') as f:
            f.write(report)
        return report

    def _format_insights(self) -> str:
        """Format insights for report."""
        return "\n".join([f"- {i}" for i in self.results.get('eda_analysis', {}).get('insights', [])])

    def _format_visualizations(self) -> str:
        """Format visualization paths."""
        return "\n".join([f"- {v}" for v in self.results.get('visualizations', {}).values()])

    def run_complete_analysis(self, data_path: Optional[str] = None) -> Dict[str, Any]:
        """Run full EDA workflow."""
        print("Starting Full EDA...")
        try:
            if data_path:
                self.load_data(data_path)
            else:
                self.create_sample_data()
            prep_results = self.run_data_preparation()
            eda_results = self.run_eda_analysis()
            visualizations = self.create_visualizations()
            report = self.generate_report()
            return {
                "preparation": prep_results,
                "eda": eda_results,
                "visualizations": visualizations,
                "report": report
            }
        except Exception as e:
            print(f"Analysis error: {e}")
            return {"error": str(e)}

def main():
    """Main function to run EDA system."""
    try:
        eda_system = EDAMultiAgentSystem()
        results = eda_system.run_complete_analysis()
        if "error" not in results:
            print("\n=== Analysis Summary ===")
            print(f"Shape: {results['preparation']['info']['shape']}")
            print("Visualizations:", list(results['visualizations'].values()))
            print("Report saved to: eda_output/eda_report.md")
        else:
            print(f"An error occurred: {results['error']}")
    except Exception as e:
        print(f"System error: {e}")
        print("Ensure GEMINI_API_KEY is set and dependencies are installed.")

if __name__ == "__main__":
    main()